# 4.2 — KNN: Every Concept Explained From Scratch
**Deep Theory + Visuals + From-Scratch Code — Nothing Skipped**

## Table of Contents
1. What is KNN — lazy learning
2. Distance metrics — Euclidean, Manhattan, Minkowski, Chebyshev
3. Why scaling is mandatory — the proof with numbers
4. The K parameter — what it controls
5. Bias-Variance tradeoff in KNN
6. Decision boundaries — visualised across K values
7. Weighted KNN — closer neighbours vote more
8. KNN for Regression — not just classification
9. Curse of Dimensionality — why KNN fails in high dimensions
10. Time & space complexity — why KNN is slow at prediction
11. Full KNN from scratch — complete implementation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Circle
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_moons, make_circles, make_classification
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report
np.random.seed(42)
print("Ready.")

---
## Concept 1 — What is KNN: Lazy Learning

### The fundamental assumption
**Similar inputs produce similar outputs.**
Two patients with similar glucose, age, BMI → likely same health outcome.
KNN never learns a formula. It just remembers all training data and looks up neighbours at prediction time.

### Lazy Learner vs Eager Learner
| | Lazy (KNN) | Eager (Logistic Regression, Decision Tree) |
|---|---|---|
| Training phase | Store data. Nothing else. | Learn weights / rules from data |
| Prediction phase | Compute distances → vote | Apply learned formula |
| Training time | O(1) — instant | Slow |
| Prediction time | O(n) — slow | O(1) — instant |

### The 5-step prediction algorithm
```
Given: training data (X_train, y_train), new point x_new, integer K

Step 1: Compute distance(x_new, every point in X_train)
Step 2: Sort all training points by distance — nearest first
Step 3: Take the top K (closest)
Step 4: Count class labels among those K
Step 5: Return majority class
```

In [ ]:
# === Concept 1: Core KNN step by step from scratch ===

np.random.seed(42)

# Small toy dataset — 2D so we can visualise
X_train_toy = np.array([
    [1, 2], [1.5, 1.8], [2, 2.5],   # class 0 — bottom-left cluster
    [6, 7], [7, 6.5], [6.5, 7.5],   # class 1 — top-right cluster
    [3.5, 4], [4, 3.5]              # class 0 — middle
], dtype=float)
y_train_toy = np.array([0,0,0, 1,1,1, 0,0])

x_new = np.array([4.5, 5.0])   # unknown point

# Step 1: compute all distances
distances = np.sqrt(np.sum((X_train_toy - x_new)**2, axis=1))

# Step 2: sort by distance
sorted_indices = np.argsort(distances)

# Step 3: pick K=3
K = 3
k_indices = sorted_indices[:K]

print("=== KNN Step-by-Step ===")
print(f"Query point: {x_new}")
print()
print("All distances:")
for i, (idx, d) in enumerate(zip(sorted_indices, distances[sorted_indices])):
    marker = " ← neighbour" if i < K else ""
    print(f"  Training point {idx}: {X_train_toy[idx]} | class={y_train_toy[idx]} | dist={d:.3f}{marker}")

# Step 4: vote
k_labels = y_train_toy[k_indices]
print(f"\nK={K} nearest labels: {k_labels}")
vals, counts = np.unique(k_labels, return_counts=True)
for v, c in zip(vals, counts):
    print(f"  Class {v}: {c} votes")

# Step 5: predict
prediction = vals[np.argmax(counts)]
print(f"\nPrediction: Class {prediction}")

In [ ]:
# Visualise the above
fig, ax = plt.subplots(figsize=(8, 6))

colors = ['steelblue' if y==0 else 'coral' for y in y_train_toy]
ax.scatter(X_train_toy[:,0], X_train_toy[:,1], c=colors, s=120, zorder=5,
           edgecolors='black', linewidths=0.8)
ax.scatter(*x_new, c='green', s=250, marker='*', zorder=6, label='Query point (unknown)')

# Lines to K nearest
for idx in k_indices:
    ax.plot([x_new[0], X_train_toy[idx,0]], [x_new[1], X_train_toy[idx,1]],
            'gray', linestyle='--', linewidth=1.5, alpha=0.8)

# Circle showing radius of K-th neighbour
radius = distances[sorted_indices[K-1]] + 0.1
circle = Circle(x_new, radius, fill=False, color='green', linestyle=':', linewidth=2)
ax.add_patch(circle)

# Legend patches
import matplotlib.patches as mpatches
ax.legend(handles=[
    mpatches.Patch(color='steelblue', label='Class 0'),
    mpatches.Patch(color='coral',     label='Class 1'),
    mpatches.Patch(color='green',     label=f'Query → Predicted Class {prediction}')
], fontsize=10)

ax.set_title(f'KNN Core: K={K}, Green star gets vote from {K} neighbours', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Concept 2 — Distance Metrics

The **choice of metric changes who your neighbours are.** 

### Euclidean (L2) — straight-line distance
$$d = \sqrt{\sum_{i}(a_i - b_i)^2}$$
- Most common default
- Sensitive to large differences in any single dimension
- Forms circular neighbourhoods

### Manhattan (L1) — city block distance
$$d = \sum_{i}|a_i - b_i|$$
- Only horizontal + vertical moves (like a taxi on a grid)
- Less sensitive to outliers
- Better in high dimensions

### Minkowski (Lp) — generalisation of both
$$d = \left(\sum_{i}|a_i - b_i|^p\right)^{1/p}$$
- p=1 → Manhattan, p=2 → Euclidean
- p is itself a tunable hyperparameter

### Chebyshev (L∞)
$$d = \max_i |a_i - b_i|$$
- Only the single largest difference matters
- Used in chess: how many king moves to reach a square?

In [ ]:
# === Concept 2: All distance metrics from scratch ===

def euclidean(a, b):
    a, b = np.array(a, float), np.array(b, float)
    return np.sqrt(np.sum((a - b)**2))

def manhattan(a, b):
    a, b = np.array(a, float), np.array(b, float)
    return np.sum(np.abs(a - b))

def minkowski(a, b, p):
    a, b = np.array(a, float), np.array(b, float)
    return np.sum(np.abs(a - b)**p)**(1/p)

def chebyshev(a, b):
    a, b = np.array(a, float), np.array(b, float)
    return np.max(np.abs(a - b))

# Two employees in Bangalore: [salary_lpa, age, experience_yrs]
emp_a = [12.0, 28, 4]
emp_b = [14.5, 35, 8]

print("Employee A:", emp_a)
print("Employee B:", emp_b)
print()
print(f"Euclidean distance:       {euclidean(emp_a, emp_b):.4f}")
print(f"Manhattan distance:       {manhattan(emp_a, emp_b):.4f}")
print(f"Minkowski (p=3):          {minkowski(emp_a, emp_b, 3):.4f}")
print(f"Chebyshev distance:       {chebyshev(emp_a, emp_b):.4f}")
print()
print("Manual Euclidean check:")
print(f"  sqrt((12-14.5)² + (28-35)² + (4-8)²)")
print(f"  = sqrt({(12-14.5)**2:.2f} + {(28-35)**2:.2f} + {(4-8)**2:.2f})")
print(f"  = sqrt({(12-14.5)**2 + (28-35)**2 + (4-8)**2:.2f})")
print(f"  = {euclidean(emp_a, emp_b):.4f}")

In [ ]:
# Visualise unit 'balls' — all points at distance=1 from origin
# Shape reveals how each metric defines 'closeness'

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
configs = [
    ('Euclidean (p=2)\nCircle',   2,  'steelblue'),
    ('Manhattan (p=1)\nDiamond',  1,  'coral'),
    ('Minkowski (p=0.5)\nStar',   0.5,'mediumseagreen'),
    ('Chebyshev (p→∞)\nSquare',   50, 'mediumpurple'),
]

for ax, (title, p, color) in zip(axes, configs):
    pts = []
    for theta in np.linspace(0, 2*np.pi, 1000):
        x, y = np.cos(theta), np.sin(theta)
        d = minkowski([x,y],[0,0], p)
        pts.append([x/d, y/d])
    pts = np.array(pts)
    ax.fill(pts[:,0], pts[:,1], alpha=0.3, color=color)
    ax.plot(pts[:,0], pts[:,1], color=color, linewidth=2.5)
    ax.set_title(title, fontsize=10)
    ax.set_xlim(-1.6,1.6); ax.set_ylim(-1.6,1.6)
    ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)

plt.suptitle('Unit Ball Shape per Metric — All points inside are "equidistant" from origin', fontsize=11)
plt.tight_layout()
plt.show()
print("Key: the shape tells you which region KNN considers 'close'.")
print("Euclidean = circular. Manhattan = diamond. Chebyshev = square.")

---
## Concept 3 — Why Scaling is Mandatory

KNN uses distance. Features with large numeric ranges dominate the calculation — not because they are more important, but because their numbers are bigger.

**StandardScaler** converts every feature to **mean=0, std=1:**
$$x_{scaled} = \frac{x - \mu}{\sigma}$$

After scaling, all features contribute proportionally to distance.

**Without scaling:** salary (₹50,000) vs age (30) → salary contributes 1,667× more to distance.  
**With scaling:** both features contribute equally regardless of original units.

In [ ]:
# === Concept 3: Prove scaling impact with real numbers ===

# 3 farmers in Kerala: [land_area_cents, annual_rainfall_mm, soil_ph]
# land_area: 10–500 cents
# rainfall:  1000–3000 mm/year
# soil_ph:   5.0–7.5
farmers = np.array([
    [250, 2100, 6.2],   # Farmer A
    [252, 1200, 6.3],   # Farmer B — nearly same land, very different rainfall
    [400, 2080, 6.1],   # Farmer C — very different land, nearly same rainfall
], dtype=float)

query_farmer = np.array([251, 2090, 6.2])   # who is nearest?

print("Query farmer: land=251 cents, rain=2090mm, ph=6.2")
print()

# Without scaling
print("=== WITHOUT SCALING ===")
for name, f in zip(['Farmer A','Farmer B','Farmer C'], farmers):
    d = euclidean(query_farmer, f)
    print(f"{name}: dist={d:.2f}  "
          f"(land_diff={abs(query_farmer[0]-f[0]):.0f}, "
          f"rain_diff={abs(query_farmer[1]-f[1]):.0f}, "
          f"ph_diff={abs(query_farmer[2]-f[2]):.2f})")

raw_dists = [euclidean(query_farmer, f) for f in farmers]
print(f"→ Nearest: Farmer {['A','B','C'][np.argmin(raw_dists)]}")
print("  PROBLEM: rainfall difference (890mm) overwhelms land and pH!")

# With scaling
print()
print("=== WITH STANDARDSCALER ===")
all_data = np.vstack([farmers, query_farmer])
scaler = StandardScaler()
all_scaled = scaler.fit_transform(all_data)
f_scaled = all_scaled[:3]
q_scaled  = all_scaled[3]

for name, fs in zip(['Farmer A','Farmer B','Farmer C'], f_scaled):
    d = euclidean(q_scaled, fs)
    print(f"{name}: scaled dist={d:.4f}")

scaled_dists = [euclidean(q_scaled, f) for f in f_scaled]
print(f"→ Nearest: Farmer {['A','B','C'][np.argmin(scaled_dists)]}")
print("  All features now contribute equally to distance.")

---
## Concept 4 — The K Parameter

K controls how many neighbours participate in the vote.

### What K really controls: the size of the 'voting region'

- **K=1**: only the single nearest point votes → extremely local decision → memorises noise
- **K=n (all training points)**: everyone votes → always predicts majority class
- **Between**: the sweet spot

### Odd K for binary classification
Always use odd K to avoid tie votes (2 class 0 vs 2 class 1 when K=4).

### Finding best K
Try K=1 to K=30. For each K, compute 5-fold cross-validated F1. Pick the K where CV F1 peaks.

In [ ]:
# === Concept 4: Effect of K on predictions ===

X_moons, y_moons = make_moons(n_samples=400, noise=0.3, random_state=42)
X_moons = StandardScaler().fit_transform(X_moons)
X_tr, X_te, y_tr, y_te = train_test_split(X_moons, y_moons, test_size=0.25,
                                            random_state=42, stratify=y_moons)

k_show = [1, 3, 7, 15, 30, len(X_tr)]
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
xx, yy = np.meshgrid(
    np.linspace(X_moons[:,0].min()-0.3, X_moons[:,0].max()+0.3, 250),
    np.linspace(X_moons[:,1].min()-0.3, X_moons[:,1].max()+0.3, 250)
)

for ax, k in zip(axes.flat, k_show):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_tr, y_tr)
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    tr_acc = knn.score(X_tr, y_tr)
    te_acc = knn.score(X_te, y_te)

    ax.contourf(xx, yy, Z, alpha=0.22, cmap='RdBu')
    ax.contour(xx, yy, Z, colors='gray', linewidths=0.6, alpha=0.6)
    ax.scatter(X_tr[y_tr==0,0], X_tr[y_tr==0,1], c='steelblue', s=18, alpha=0.7)
    ax.scatter(X_tr[y_tr==1,0], X_tr[y_tr==1,1], c='coral',     s=18, alpha=0.7)

    label = str(k) if k < len(X_tr) else f"{k} (all points)"
    verdict = ("Overfitting" if tr_acc - te_acc > 0.05
               else "Underfitting" if te_acc < 0.80
               else "Good fit")
    ax.set_title(f'K={label}\nTrain={tr_acc*100:.1f}%  Test={te_acc*100:.1f}%  [{verdict}]',
                 fontsize=9)
    ax.grid(True, alpha=0.2)

plt.suptitle('Effect of K on Decision Boundary', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# K selection curve — CV F1 vs K
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
k_range = range(1, 31)
train_acc, test_acc, cv_f1 = [], [], []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_tr, y_tr)
    train_acc.append(knn.score(X_tr, y_tr))
    test_acc.append(knn.score(X_te, y_te))
    cv_f1.append(cross_val_score(knn, X_tr, y_tr, cv=cv, scoring='f1').mean())

best_k = list(k_range)[np.argmax(cv_f1)]

plt.figure(figsize=(10, 4))
plt.plot(k_range, train_acc, label='Train accuracy', color='steelblue')
plt.plot(k_range, test_acc,  label='Test accuracy',  color='coral')
plt.plot(k_range, cv_f1,     label='CV F1 (reliable)', color='green', linestyle='--')
plt.axvline(best_k, color='black', linestyle=':', label=f'Best K={best_k}')
plt.title('K Selection Curve — Pick K at Peak CV F1')
plt.xlabel('K'); plt.ylabel('Score'); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f"Best K = {best_k} | CV F1 = {max(cv_f1):.4f}")
print("Always use cross-validated score — not test accuracy — to pick K.")
print("Test accuracy should be touched only ONCE at the very end.")

---
## Concept 5 — Bias-Variance Tradeoff in KNN

This is the most important concept in all of ML and KNN makes it very concrete.

### What is bias?
The error from wrong assumptions. A model with high bias is **too simple** — it misses real patterns.

### What is variance?
The error from sensitivity to training data. A model with high variance changes dramatically when the training data changes slightly.

### KNN and the tradeoff
| K | Bias | Variance | Behaviour |
|---|---|---|---|
| K=1 | Very low | Very high | Memorises every point — overfits |
| K=n | Very high | Very low | Always predicts majority — underfits |
| K=optimal | Balanced | Balanced | Generalises well |

**The sweet spot:** K where test error is minimised — found by cross-validation.

In [ ]:
# === Concept 5: Bias-Variance tradeoff visualised ===

# Generate many training sets, fit KNN with different K, plot prediction variance
np.random.seed(0)
n_datasets = 30

# True function: a sine wave (regression setting to visualise clearly)
X_range = np.linspace(0, 6, 300).reshape(-1, 1)
y_true   = np.sin(X_range).ravel()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, k in zip(axes, [1, 7, 50]):
    predictions_list = []
    for _ in range(n_datasets):
        # Each dataset is slightly different (simulating different training sets)
        X_s = np.random.uniform(0, 6, 40).reshape(-1,1)
        y_s = np.sin(X_s).ravel() + np.random.normal(0, 0.3, 40)
        knn_r = KNeighborsRegressor(n_neighbors=k)
        knn_r.fit(X_s, y_s)
        predictions_list.append(knn_r.predict(X_range))

    preds = np.array(predictions_list)
    mean_pred = preds.mean(axis=0)
    std_pred  = preds.std(axis=0)

    # Plot individual model predictions (shows variance)
    for pred in predictions_list[:8]:
        ax.plot(X_range, pred, color='steelblue', alpha=0.15, linewidth=1)

    ax.plot(X_range, y_true,    'black',  linewidth=2,   label='True function')
    ax.plot(X_range, mean_pred, 'coral',  linewidth=2,   label='Mean prediction')
    ax.fill_between(X_range.ravel(),
                    mean_pred - std_pred,
                    mean_pred + std_pred,
                    alpha=0.2, color='coral', label='±1 std (variance)')

    bias_sq = np.mean((mean_pred - y_true)**2)
    variance = np.mean(std_pred**2)
    ax.set_title(f'K={k}\nBias²={bias_sq:.3f}  Variance={variance:.3f}', fontsize=11)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    ax.set_ylim(-2, 2)

plt.suptitle('Bias-Variance Tradeoff in KNN\n'
             'K=1: low bias, high variance (wiggly) | K=50: high bias, low variance (flat)',
             fontsize=11)
plt.tight_layout()
plt.show()

---
## Concept 6 — Weighted KNN

Standard KNN: all K neighbours get **equal vote**.

Weighted KNN: **closer neighbours get more weight.**

Weight formula (inverse distance):
$$w_i = \frac{1}{d_i}$$

If one neighbour is at distance 0.1 and another at distance 5.0 → the close one gets 50× more weight.

**When does this help?**
- When K is large and distant neighbours add noise
- When classes overlap in some regions
- sklearn: `weights='distance'` (default is `weights='uniform'`)

In [ ]:
# === Concept 6: Uniform vs Weighted KNN ===

# Dataset where weighted helps: overlapping classes near boundary
np.random.seed(5)
X_w = np.vstack([
    np.random.randn(80,2) + [0, 0],
    np.random.randn(80,2) + [2, 2]
])
y_w = np.array([0]*80 + [1]*80)
X_w = StandardScaler().fit_transform(X_w)
X_wtr, X_wte, y_wtr, y_wte = train_test_split(X_w, y_w, test_size=0.25,
                                                 random_state=42, stratify=y_w)
xx, yy = np.meshgrid(
    np.linspace(X_w[:,0].min()-0.3, X_w[:,0].max()+0.3, 200),
    np.linspace(X_w[:,1].min()-0.3, X_w[:,1].max()+0.3, 200)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, weight, title in zip(axes,
    ['uniform', 'distance'],
    ['Uniform weights (all K neighbours equal)',
     'Distance weights (closer = more weight)']):

    knn = KNeighborsClassifier(n_neighbors=7, weights=weight)
    knn.fit(X_wtr, y_wtr)
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    acc = knn.score(X_wte, y_wte)

    ax.contourf(xx, yy, Z, alpha=0.2, cmap='RdBu')
    ax.scatter(X_wtr[y_wtr==0,0], X_wtr[y_wtr==0,1], c='steelblue', s=25, alpha=0.7)
    ax.scatter(X_wtr[y_wtr==1,0], X_wtr[y_wtr==1,1], c='coral',     s=25, alpha=0.7)
    ax.set_title(f'{title}\nTest accuracy: {acc*100:.1f}%', fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('Weighted vs Uniform KNN', fontsize=12)
plt.tight_layout(); plt.show()

# Show the weight calculation manually
print("=== How distance weighting works ===")
print("5 neighbours with distances: [0.1, 0.3, 0.5, 2.0, 4.0]")
dists = np.array([0.1, 0.3, 0.5, 2.0, 4.0])
labels = np.array([1, 1, 0, 0, 0])
weights = 1 / dists
print(f"Weights (1/d):              {weights.round(3)}")
print(f"Labels:                     {labels}")
weighted_votes = {c: weights[labels==c].sum() for c in [0,1]}
print(f"Weighted vote class 0: {weighted_votes[0]:.3f}")
print(f"Weighted vote class 1: {weighted_votes[1]:.3f}")
print(f"Prediction: Class {max(weighted_votes, key=weighted_votes.get)}")
print("(With uniform: 3 zeros vs 2 ones → Class 0. Weighting flips the result!)")

---
## Concept 7 — KNN for Regression

KNN works for regression too. Instead of majority vote → **average the K neighbours' values.**

$$\hat{y} = \frac{1}{K} \sum_{i \in K\_neighbours} y_i$$

With distance weighting:
$$\hat{y} = \frac{\sum_{i} w_i \cdot y_i}{\sum_{i} w_i}$$

In [ ]:
# === Concept 7: KNN Regression from scratch and with sklearn ===

# From scratch
def knn_regress(X_train, y_train, x_new, k=3, weighted=False):
    dists = np.sqrt(np.sum((X_train - x_new)**2, axis=1))
    k_idx = np.argsort(dists)[:k]
    k_y   = y_train[k_idx]
    k_d   = dists[k_idx]
    if weighted:
        w = 1.0 / (k_d + 1e-8)   # tiny epsilon to avoid div-by-zero
        return np.dot(w, k_y) / w.sum()
    else:
        return k_y.mean()

# 1D dataset: study hours → exam score
X_reg = np.linspace(1, 10, 80).reshape(-1, 1)
y_reg = 40 + 5*X_reg.ravel() + 3*np.sin(X_reg.ravel()*2) + np.random.randn(80)*4

X_range_1d = np.linspace(1, 10, 200).reshape(-1, 1)

plt.figure(figsize=(14, 5))
colors = ['coral', 'steelblue', 'mediumseagreen']
for i, k in enumerate([1, 5, 20]):
    reg = KNeighborsRegressor(n_neighbors=k, weights='uniform')
    reg.fit(X_reg, y_reg)
    y_hat = reg.predict(X_range_1d)

    plt.subplot(1, 3, i+1)
    plt.scatter(X_reg, y_reg, c='black', s=20, alpha=0.5, label='Training data')
    plt.plot(X_range_1d, y_hat, color=colors[i], linewidth=2.5, label=f'KNN K={k}')
    plt.title(f'KNN Regression K={k}', fontsize=11)
    plt.xlabel('Study hours'); plt.ylabel('Exam score')
    plt.legend(fontsize=9); plt.grid(True, alpha=0.3)

plt.suptitle('KNN Regression — K controls smoothness of prediction curve', fontsize=12)
plt.tight_layout(); plt.show()

# Demonstrate from-scratch
print("From-scratch KNN Regression:")
print(f"Predicting exam score for 6 study hours:")
X_s_arr = X_reg.ravel()
print(f"  K=1 (uniform):  {knn_regress(X_s_arr.reshape(-1,1), y_reg, [[6]], 1):.1f}")
print(f"  K=5 (uniform):  {knn_regress(X_s_arr.reshape(-1,1), y_reg, [[6]], 5):.1f}")
print(f"  K=5 (weighted): {knn_regress(X_s_arr.reshape(-1,1), y_reg, [[6]], 5, weighted=True):.1f}")

---
## Concept 8 — Curse of Dimensionality

This is the most important limitation of KNN.

### What happens as dimensions increase?

In 1D: to cover 10% of the data range, you need 10% of the axis.  
In 2D: to cover 10% of the area, you need 31.6% of each axis.  
In 10D: to cover 10% of the volume, you need **79.4%** of each axis.

**In high dimensions, "nearest" neighbours are no longer meaningfully close.**
All points become approximately equidistant → voting becomes meaningless.

$$\text{edge length to cover fraction } f \text{ in } d \text{ dimensions} = f^{1/d}$$

### The distance concentration problem
As dimensions grow, the ratio of max-to-min distance approaches 1.  
If all distances are similar, the concept of 'nearest' breaks down.

In [ ]:
# === Concept 8: Curse of Dimensionality ===

# Part A: Edge length needed to cover 10% of data
f = 0.10   # we want to cover 10% of the data
dims = np.arange(1, 101)
edge_lengths = f**(1/dims)

plt.figure(figsize=(10, 4))
plt.plot(dims, edge_lengths * 100, color='coral', linewidth=2)
plt.axhline(10,  color='gray',  linestyle='--', alpha=0.7, label='10% of axis')
plt.axhline(79,  color='black', linestyle=':', alpha=0.7,  label='79% at d=10')
plt.xlabel('Number of Dimensions')
plt.ylabel('% of each axis needed to cover 10% of data')
plt.title('Curse of Dimensionality\n'          'In 10D, you need 79% of each axis just to cover 10% of the data volume')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# Part B: Distance concentration
print("=== Distance Concentration (all distances become equal) ===")
n_pts = 500
for d in [2, 5, 10, 50, 100]:
    pts   = np.random.randn(n_pts, d)
    dists = np.sqrt(np.sum((pts - pts[0])**2, axis=1)[1:])
    ratio = dists.max() / dists.min()
    print(f"  d={d:3d}: min_dist={dists.min():.2f}  max_dist={dists.max():.2f}  "
          f"ratio={ratio:.2f}  spread={dists.std():.2f}")

print()
print("As d increases: max/min ratio → 1.0")
print("All points become equally 'far' → nearest neighbour is meaningless")
print("Rule of thumb: KNN struggles when n_features > 10-15")

---
## Concept 9 — Time & Space Complexity

### Why KNN is slow at prediction time

| Operation | Complexity | Explanation |
|---|---|---|
| Training | O(1) | Just store data — nothing computed |
| Prediction (1 point) | O(n × d) | Compare to every training point across all d features |
| Prediction (m points) | O(m × n × d) | Scales with test set size too |
| Memory | O(n × d) | Must store entire training set |

Where n = training samples, d = features, m = test samples.

**For 1 million training points:** predicting 1 query = 1 million distance computations.

### KD-Tree and Ball-Tree (sklearn's trick)
sklearn doesn't do brute-force search by default. It builds spatial index structures:
- **KD-Tree**: partition space by axis-aligned hyperplanes. Fast for d < 20.
- **Ball-Tree**: partition space into nested hyperspheres. Better for curved manifolds.
These reduce prediction to O(log n) on average — but still degrade in high dimensions.

In [ ]:
# === Concept 9: Measure actual prediction time vs training size ===
import time

dimensions = 5
times_brute  = []
times_kd     = []
sizes = [100, 500, 1000, 5000, 10000]

for n in sizes:
    X_t = np.random.randn(n, dimensions)
    y_t = np.random.randint(0, 2, n)
    x_q = np.random.randn(1, dimensions)

    # Brute force
    knn_b = KNeighborsClassifier(n_neighbors=5, algorithm='brute')
    knn_b.fit(X_t, y_t)
    t0 = time.perf_counter()
    for _ in range(50): knn_b.predict(x_q)
    times_brute.append((time.perf_counter()-t0)/50 * 1000)

    # KD-Tree
    knn_k = KNeighborsClassifier(n_neighbors=5, algorithm='kd_tree')
    knn_k.fit(X_t, y_t)
    t0 = time.perf_counter()
    for _ in range(50): knn_k.predict(x_q)
    times_kd.append((time.perf_counter()-t0)/50 * 1000)

plt.figure(figsize=(9, 4))
plt.plot(sizes, times_brute, color='coral',     marker='o', label='Brute force O(n)')
plt.plot(sizes, times_kd,    color='steelblue', marker='s', label='KD-Tree O(log n)')
plt.xlabel('Training set size (n)'); plt.ylabel('Prediction time (ms)')
plt.title('Prediction Time vs Training Size\nBrute force grows linearly. KD-Tree is faster.')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

print("Takeaway: for large datasets, use algorithm='kd_tree' or 'ball_tree'.")
print("For n > 100,000 — consider a different algorithm entirely.")

---
## Concept 10 — Full KNN from Scratch (Complete Implementation)

Everything we've covered — put together into one clean class that mirrors sklearn's API.

In [ ]:
# === Concept 10: Full KNN Classifier from scratch ===

class KNNClassifier:
    """
    K-Nearest Neighbors Classifier built from scratch.
    Supports: Euclidean, Manhattan, Minkowski distance.
    Supports: uniform and distance-based weighting.
    """

    def __init__(self, k=5, metric='euclidean', p=2, weights='uniform'):
        self.k       = k          # number of neighbours
        self.metric  = metric     # 'euclidean', 'manhattan', 'minkowski'
        self.p       = p          # for minkowski
        self.weights = weights    # 'uniform' or 'distance'

    def fit(self, X, y):
        """Training = store data. Nothing else."""
        self.X_train = np.array(X, dtype=float)
        self.y_train = np.array(y)
        self.classes_ = np.unique(y)
        return self

    def _distance(self, a, b):
        if self.metric == 'euclidean':
            return np.sqrt(np.sum((a - b)**2, axis=1))
        elif self.metric == 'manhattan':
            return np.sum(np.abs(a - b), axis=1)
        elif self.metric == 'minkowski':
            return np.sum(np.abs(a - b)**self.p, axis=1)**(1/self.p)
        else:
            raise ValueError(f"Unknown metric: {self.metric}")

    def _predict_one(self, x):
        """Predict class for one sample."""
        # Compute distances from x to all training points
        dists = self._distance(self.X_train, x)

        # Get K nearest indices and their distances
        k_idx   = np.argsort(dists)[:self.k]
        k_dists = dists[k_idx]
        k_labels= self.y_train[k_idx]

        if self.weights == 'uniform':
            # Each neighbour gets 1 vote
            vals, counts = np.unique(k_labels, return_counts=True)
            return vals[np.argmax(counts)]
        else:
            # Closer neighbours get higher weight: w = 1/d
            weighted_votes = {}
            for label, d in zip(k_labels, k_dists):
                w = 1.0 / (d + 1e-8)
                weighted_votes[label] = weighted_votes.get(label, 0) + w
            return max(weighted_votes, key=weighted_votes.get)

    def predict(self, X):
        return np.array([self._predict_one(x) for x in np.array(X, dtype=float)])

    def predict_proba(self, X):
        probas = []
        for x in np.array(X, dtype=float):
            dists = self._distance(self.X_train, x)
            k_idx = np.argsort(dists)[:self.k]
            k_labels = self.y_train[k_idx]
            proba = [(k_labels == c).mean() for c in self.classes_]
            probas.append(proba)
        return np.array(probas)

    def score(self, X, y):
        return (self.predict(X) == np.array(y)).mean()


# ── Test it ──
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
X_c, y_c = data.data[:, :5], data.target   # use first 5 features

scaler = StandardScaler()
X_c_s = scaler.fit_transform(X_c)
X_ctr, X_cte, y_ctr, y_cte = train_test_split(X_c_s, y_c,
                                                 test_size=0.2, random_state=42)

# Our implementation
my_knn = KNNClassifier(k=7, metric='euclidean', weights='uniform')
my_knn.fit(X_ctr, y_ctr)
my_acc = my_knn.score(X_cte, y_cte)

# sklearn for comparison
sk_knn = KNeighborsClassifier(n_neighbors=7, metric='euclidean', weights='uniform')
sk_knn.fit(X_ctr, y_ctr)
sk_acc = sk_knn.score(X_cte, y_cte)

print("=== KNN From Scratch vs sklearn ===")
print(f"Our KNN accuracy:     {my_acc*100:.2f}%")
print(f"sklearn KNN accuracy: {sk_acc*100:.2f}%")
print("They should be identical or within rounding error.")
print()
print("Predictions match:", np.array_equal(my_knn.predict(X_cte[:10]),
                                             sk_knn.predict(X_cte[:10])))

---
## Summary — Every KNN Concept at a Glance

| Concept | Key point |
|---|---|
| Lazy learner | No training. Stores all data. Computes distances at prediction. |
| Euclidean | √Σ(aᵢ-bᵢ)² — straight line. Default choice. |
| Manhattan | Σ|aᵢ-bᵢ| — city blocks. Better for high dims. |
| Minkowski | Generalises both. p=1→Manhattan, p=2→Euclidean. |
| Scaling | **Mandatory.** Large-scale features dominate distances otherwise. |
| K=small | Low bias, high variance. Overfits. Jagged boundary. |
| K=large | High bias, low variance. Underfits. Smooth boundary. |
| Best K | Cross-validated F1. Always odd for binary. |
| Weighted KNN | `weights='distance'`. Closer neighbours vote more. |
| Regression | Average of K neighbours' values instead of vote. |
| Curse of dim | In high dimensions, all distances become similar → useless. |
| Complexity | Train O(1). Predict O(n×d) — slow for large n. |
| KD/Ball tree | sklearn's index structures. Reduce predict to O(log n). |

### When to use KNN
- Small-to-medium datasets (n < 50,000)
- Non-linear decision boundary needed
- Quick baseline before trying complex models
- When you can't assume anything about data distribution

### When NOT to use KNN
- Large datasets (slow prediction)
- More than ~15 features (curse of dimensionality)
- When you need feature importance or model explanation
- Real-time prediction with strict latency requirements